In [6]:
import json
import os
from typing import Dict, Any, List, Optional

from pathlib import Path


class WorkflowMinimizer:
    """
    Minimizes agent workflow JSON files by removing verbose content while preserving
    essential workflow information.
    """

    def __init__(
        self,
        remove_page_snapshots: bool = True,
        remove_usage_details: bool = True,
        remove_timestamps: bool = False,
        remove_vendor_info: bool = True,
        preserve_errors: bool = True,
        preserve_console_output: bool = True,
    ):
        """
        Initialize the minimizer with configuration options.

        Args:
            remove_page_snapshots: Remove detailed DOM snapshots (major size reduction)
            remove_usage_details: Remove token usage statistics
            remove_timestamps: Remove timestamp fields
            remove_vendor_info: Remove vendor_details and vendor_id
            preserve_errors: Keep error messages for debugging
            preserve_console_output: Keep console logs and outputs
        """
        self.remove_page_snapshots = remove_page_snapshots
        self.remove_usage_details = remove_usage_details
        self.remove_timestamps = remove_timestamps
        self.remove_vendor_info = remove_vendor_info
        self.preserve_errors = preserve_errors
        self.preserve_console_output = preserve_console_output

    def _clean_message_part(self, part: Dict[str, Any]) -> Dict[str, Any]:
        """Clean an individual message part."""
        cleaned = {}

        for key, value in part.items():
            # Always keep these essential fields
            if key in ["tool_name", "args", "part_kind", "tool_call_id"]:
                cleaned[key] = value

            # Handle content field - keep but potentially clean page snapshots
            elif key == "content":
                if isinstance(value, str) and self.remove_page_snapshots:
                    # Remove page snapshots but keep other content
                    if "### Page state" in value and "- Page Snapshot:" in value:
                        lines = value.split("\n")
                        filtered_lines = []
                        in_snapshot = False

                        for line in lines:
                            if "- Page Snapshot:" in line:
                                in_snapshot = True
                                filtered_lines.append(line)
                                filtered_lines.append(
                                    "    [Page snapshot removed for size optimization]"
                                )
                                continue
                            elif in_snapshot and line.strip() == "```":
                                in_snapshot = False
                                filtered_lines.append(line)
                                continue
                            elif not in_snapshot:
                                filtered_lines.append(line)

                        cleaned[key] = "\n".join(filtered_lines)
                    else:
                        cleaned[key] = value
                else:
                    cleaned[key] = value

            # Handle timestamps
            elif key == "timestamp":
                if not self.remove_timestamps:
                    cleaned[key] = value

            # Handle vendor info
            elif key in ["vendor_details", "vendor_id"]:
                if not self.remove_vendor_info:
                    cleaned[key] = value

            # Handle usage details
            elif key == "usage":
                if not self.remove_usage_details:
                    # Keep only high-level usage info
                    if isinstance(value, dict):
                        simplified_usage = {
                            "total_tokens": value.get("total_tokens"),
                            "request_tokens": value.get("request_tokens"),
                            "response_tokens": value.get("response_tokens"),
                        }
                        # Remove None values
                        cleaned[key] = {
                            k: v for k, v in simplified_usage.items() if v is not None
                        }
                    else:
                        cleaned[key] = value

            # Handle errors and console output
            elif "error" in key.lower() or "console" in key.lower():
                if self.preserve_errors or self.preserve_console_output:
                    cleaned[key] = value

            # Handle metadata
            elif key == "metadata":
                # Only keep non-null metadata
                if value is not None:
                    cleaned[key] = value

            # Keep other important fields
            else:
                cleaned[key] = value

        return cleaned

    def _extract_workflow_summary(
        self, messages: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """Extract a high-level workflow summary."""
        workflow = []

        for message in messages:
            if message.get("kind") == "response":
                for part in message.get("parts", []):
                    if part.get("part_kind") == "tool-call":
                        tool_name = part.get("tool_name")
                        args = part.get("args", {})

                        workflow_step = {
                            "tool": tool_name,
                        }

                        # Add relevant args based on tool type
                        if tool_name == "browser_navigate":
                            workflow_step["url"] = args.get("url")
                        elif tool_name == "browser_click":
                            workflow_step["element"] = args.get("element")
                        elif tool_name == "browser_type":
                            workflow_step["text"] = args.get("text")
                            workflow_step["element"] = args.get("element")
                        elif tool_name == "final_result":
                            workflow_step["result"] = "completed"

                        workflow.append(workflow_step)

        return workflow

    def minimize_json(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Minimize a workflow JSON while preserving essential information.

        Args:
            data: The original JSON data

        Returns:
            Minimized JSON data
        """
        minimized = {}

        # Always preserve the output
        if "output" in data:
            minimized["output"] = data["output"]

        # Process messages
        if "messages" in data:
            minimized_messages = []

            for message in data["messages"]:
                minimized_message = {}

                # Keep essential message fields
                for key in ["kind", "instructions"]:
                    if key in message and message[key] is not None:
                        minimized_message[key] = message[key]

                # Clean message parts
                if "parts" in message:
                    minimized_parts = []
                    for part in message["parts"]:
                        cleaned_part = self._clean_message_part(part)
                        if cleaned_part:  # Only add non-empty parts
                            minimized_parts.append(cleaned_part)

                    if minimized_parts:
                        minimized_message["parts"] = minimized_parts

                # Keep usage info if configured
                if "usage" in message and not self.remove_usage_details:
                    usage = message["usage"]
                    if isinstance(usage, dict):
                        minimized_message["usage"] = {
                            "total_tokens": usage.get("total_tokens"),
                            "request_tokens": usage.get("request_tokens"),
                            "response_tokens": usage.get("response_tokens"),
                        }

                if minimized_message:
                    minimized_messages.append(minimized_message)

            minimized["messages"] = minimized_messages

            # Add workflow summary
            minimized["workflow_summary"] = self._extract_workflow_summary(
                data["messages"]
            )

        return minimized

    def process_file(
        self, input_file: str, output_file: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Process a single JSON file.

        Args:
            input_file: Path to input JSON file
            output_file: Path to output file (optional, defaults to input_file with _minimized suffix)

        Returns:
            Statistics about the minimization process
        """
        input_path = Path(input_file)

        if output_file is None:
            output_file = (
                input_path.parent / f"{input_path.stem}_minimized{input_path.suffix}"
            )

        # Read original file
        with open(input_file, "r", encoding="utf-8") as f:
            original_data = json.load(f)

        # Get original size
        original_size = os.path.getsize(input_file)

        # Minimize
        minimized_data = self.minimize_json(original_data)

        # Write minimized file
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(minimized_data, f, indent=2, ensure_ascii=False)

        # Get new size
        new_size = os.path.getsize(output_file)

        return {
            "original_size": original_size,
            "new_size": new_size,
            "reduction_percent": ((original_size - new_size) / original_size) * 100,
            "output_file": str(output_file),
        }

    def process_directory(
        self, directory: str, pattern: str = "*.json"
    ) -> List[Dict[str, Any]]:
        """
        Process all JSON files in a directory.

        Args:
            directory: Directory path
            pattern: File pattern to match

        Returns:
            List of processing statistics for each file
        """
        results = []
        directory_path = Path(directory)

        for file_path in directory_path.glob(pattern):
            if "_minimized" not in file_path.name:  # Skip already processed files
                try:
                    result = self.process_file(str(file_path))
                    result["input_file"] = str(file_path)
                    results.append(result)
                    print(
                        f"Processed {file_path.name}: {result['reduction_percent']:.1f}% reduction"
                    )
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")

        return results


def process_nested_results(
    minimizer: WorkflowMinimizer,
    base_dir: str,
    input_filename: str = "results.json",
    output_filename: str = "reduced_results.json",
) -> List[Dict[str, Any]]:
    """
    Process JSON files in nested directory structure like data/traces/*/results.json

    This function is separated from the WorkflowMinimizer class to provide a standalone
    utility for processing specific directory structures without coupling the core
    minimization logic to particular file organization patterns.

    Args:
        minimizer: WorkflowMinimizer instance to use for processing
        base_dir: Base directory to search (e.g., "data/traces")
        input_filename: Name of input files to process
        output_filename: Name of output files to create

    Returns:
        List of processing statistics for each file
    """
    results = []
    base_path = Path(base_dir)

    # Find all files matching the pattern in subdirectories
    pattern = f"*/{input_filename}"
    for file_path in base_path.glob(pattern):
        try:
            # Create output path in same directory
            output_path = file_path.parent / output_filename

            result = minimizer.process_file(str(file_path), str(output_path))
            result["input_file"] = str(file_path)
            result["subdirectory"] = file_path.parent.name
            results.append(result)

            print(
                f"Processed {file_path.parent.name}/{input_filename}: {result['reduction_percent']:.1f}% reduction"
            )

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    return results


def print_processing_summary(results: List[Dict[str, Any]]) -> None:
    """
    Print a summary of processing results.

    Args:
        results: List of processing statistics from minimization operations
    """
    if not results:
        print("No files were processed.")
        return

    total_original = sum(r["original_size"] for r in results)
    total_new = sum(r["new_size"] for r in results)
    avg_reduction = sum(r["reduction_percent"] for r in results) / len(results)

    print("\n=== Processing Summary ===")
    print(f"Files processed: {len(results)}")
    print(f"Average reduction: {avg_reduction:.1f}%")
    print(
        f"Total size reduction: {((total_original - total_new) / total_original) * 100:.1f}%"
    )
    print(f"Original total: {total_original:,} bytes")
    print(f"New total: {total_new:,} bytes")
    print(f"Space saved: {total_original - total_new:,} bytes")


def process_custom_structure(
    base_directory: str = "../data/traces",
) -> List[Dict[str, Any]]:
    """
    Standalone function to process your specific directory structure.

    This function creates a minimizer with sensible defaults and processes
    a nested directory structure. It's designed to be easily callable from
    other scripts or as a command-line utility.

    Args:
        base_directory: Base directory containing the nested structure

    Returns:
        List of processing statistics for each file processed
    """
    minimizer = WorkflowMinimizer(
        remove_page_snapshots=True,
        remove_usage_details=True,
        remove_timestamps=False,
        remove_vendor_info=True,
        preserve_errors=True,
        preserve_console_output=True,
    )

    results = process_nested_results(
        minimizer=minimizer,
        base_dir=base_directory,
        input_filename="results.json",
        output_filename="reduced_results.json",
    )

    return results


def main():
    """Example usage of the WorkflowMinimizer with improved structure."""

    # Create minimizer with default aggressive settings
    minimizer = WorkflowMinimizer(
        remove_page_snapshots=True,  # Major size reduction
        remove_usage_details=True,  # Remove token statistics
        remove_timestamps=False,  # Keep timestamps for workflow analysis
        remove_vendor_info=True,  # Remove vendor details
        preserve_errors=True,  # Keep error messages
        preserve_console_output=True,  # Keep console logs
    )

    # Process nested directory structure: data/traces/*/results.json
    results = process_nested_results(
        minimizer=minimizer,
        base_dir="../data/traces",
        input_filename="result.json",
        output_filename="reduced_results.json",
    )

    # Print summary statistics using the new helper function
    print_processing_summary(results)

    if not results:
        print("No files found matching pattern data/traces/*/results.json")


if __name__ == "__main__":
    main()

Processed north_kesteven_district_council/result.json: 87.0% reduction
Processed mid_sussex_district_council/result.json: 91.7% reduction
Processed buckinghamshire_council/result.json: 77.0% reduction
Processed north_west_leicestershire_district_council/result.json: 89.6% reduction
Processed warrington_borough_council/result.json: 92.7% reduction
Processed east_lindsey_district_council/result.json: 79.7% reduction
Processed torbay_council/result.json: 35.3% reduction
Processed colchester_borough_council/result.json: 80.8% reduction
Processed woking_borough_council/result.json: 87.1% reduction
Processed fareham_borough_council/result.json: 87.6% reduction
Processed huntingdonshire_district_council/result.json: 87.3% reduction
Processed gosport_borough_council/result.json: 85.9% reduction
Processed north_hertfordshire_district_council/result.json: 87.7% reduction
Processed renfrewshire_council/result.json: 28.6% reduction
Processed salford_city_council/result.json: 87.4% reduction
Proces